<a href="https://colab.research.google.com/github/DeepthiManthapuram/Deep_Learning/blob/main/Translator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Build a mini English - French translator using LSTM Encoder-Decoder

Classic Seq2Seq use-case

Covers tokenization, padding, training, inference
Easy to extend later to atluntion + transformers

1. Load dataset
2. Preprocess text (tokenize, pad)
3. Build Encoder (LSTM)
4. Build Decoder (LSTM)
5. Train model
6. Inference (predict new sentence)

Input Sentence

Encoder LSTM

Final Hidden State + Cell State

Context vector

Output LSTM

Generate output word by word

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# Sample dataset
data = [
    ("i am learning", "je suis en train d apprendre"),
    ("he is running", "il court"),
    ("she is happy", "elle est heureuse"),
    ("i am happy", "je suis heureux")
]

input_text = [x[0] for x in data]
target_text = ["<start>" + x[1] + "<end>" for x in data]

# Tokenization

In [ ]:
#Encoder Tokenizer

input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_text)
input_sequences = input_tokenizer.texts_to_sequences(input_text)

#Decoder Tokenizer

target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_text)
target_sequences = target_tokenizer.texts_to_sequences(target_text)




# Padding

In [ ]:
max_input_len = max([len(seq) for seq in input_sequences])
max_target_len = max([len(seq) for seq in target_sequences])

encoder_input_data = pad_sequences(input_sequences, maxlen=max_input_len, padding='post')
decoder_input_data = pad_sequences(target_sequences, maxlen=max_target_len, padding='post')

# Create Decoder Target (Shifted)

Important teaching point:
Decoder input = ...

## Decoder output = ...

In [ ]:
decoder_target_data = np.zeros_like(decoder_input_data)

for i in range(len(data)):
  decoder_target_data[i, :-1] = decoder_input_data[i, 1:]

### Regenerating Data for Correct Training

It appears that the input sequences used for training might have been incorrectly generated, leading to the model confusing similar input phrases. To resolve this, we will explicitly re-generate all input and target sequence data to ensure accuracy before retraining the model.

In [ ]:
# Re-generate input and target sequences to ensure correctness
# This step re-applies the tokenizers to the original text data.
input_sequences = input_tokenizer.texts_to_sequences(input_text)
target_sequences = target_tokenizer.texts_to_sequences(target_text)

# Recalculate max lengths in case they were affected (though typically stable)
max_input_len = max([len(seq) for seq in input_sequences])
max_target_len = max([len(seq) for seq in target_sequences])

# Re-generate padded data for encoder and decoder inputs
encoder_input_data = pad_sequences(input_sequences, maxlen=max_input_len, padding='post')
decoder_input_data = pad_sequences(target_sequences, maxlen=max_target_len, padding='post')

# Re-generate decoder target data based on the corrected decoder_input_data
decoder_target_data = np.zeros_like(decoder_input_data)
for i in range(len(data)):
  decoder_target_data[i, :-1] = decoder_input_data[i, 1:]

print("Regenerated encoder_input_data (first 5 rows):\n", encoder_input_data[:5])
print("\nRegenerated decoder_target_data (first 5 rows):\n", decoder_target_data[:5])

print("\nData regeneration complete. Please re-run the model compilation, training, and inference steps from the notebook to apply these corrections.")

Regenerated encoder_input_data (first 5 rows):
 [[1 2 5]
 [6 3 7]
 [8 3 4]
 [1 2 4]]

Regenerated decoder_target_data (first 5 rows):
 [[ 3  4  5  6  7  8  2  0]
 [ 9 10  2  0  0  0  0  0]
 [11 12 13  2  0  0  0  0]
 [ 3  4 14  2  0  0  0  0]]

Data regeneration complete. Please re-run the model compilation, training, and inference steps from the notebook to apply these corrections.


# Build Model

Parameters

In [ ]:
vocab_size_input = len(input_tokenizer.word_index) + 1
vocab_size_target = len(target_tokenizer.word_index) + 1
embedding_dim=50
latent_dim=100

# Encoder

In [ ]:
encoder_input = Input(shape=(None, ))
enc_emb = Embedding(vocab_size_input, embedding_dim)(encoder_input)

encoder_lstm = LSTM(latent_dim, return_state = True)
_, state_h, state_c = encoder_lstm(enc_emb)

encoder_states = [state_h, state_c]
print(encoder_states)

[<KerasTensor shape=(None, 100), dtype=float32, sparse=False, ragged=False, name=keras_tensor_59>, <KerasTensor shape=(None, 100), dtype=float32, sparse=False, ragged=False, name=keras_tensor_60>]


In [ ]:
decoder_input = Input(shape=(None, ))
dec_emb_layer = Embedding(vocab_size_target, embedding_dim) # Use vocab_size_target and define the layer
dec_emb = dec_emb_layer(decoder_input) # Call the layer with decoder_input

decoder_lstm = LSTM(latent_dim, return_sequences = True, return_state = True) # Fix typo
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state = encoder_states)

decoder_dense = Dense(vocab_size_target, activation = 'softmax')
decoder_outputs = decoder_dense(decoder_outputs)
print(decoder_outputs)

<KerasTensor shape=(None, None, 15), dtype=float32, sparse=False, ragged=False, name=keras_tensor_66>


# Compile Model

In [ ]:
model = Model([encoder_input, decoder_input], decoder_outputs)
model.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy')

# Train Model

In [ ]:
model.fit([
    encoder_input_data, decoder_input_data],
          decoder_target_data,
          batch_size=1,
          epochs=200,
          validation_split = 0.2
          )

Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 173ms/step - loss: 2.7086 - val_loss: 2.6885
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 2.6723 - val_loss: 2.6599
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 2.6453 - val_loss: 2.6210
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 2.6002 - val_loss: 2.5669
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 2.5320 - val_loss: 2.4859
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 2.4186 - val_loss: 2.3625
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 2.2489 - val_loss: 2.1583
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 1.9863 - val_loss: 1.8456
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 1.7499 - val_loss: 1.5759
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 1.7059 - val_loss: 1.4761
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 1.7052 - val_loss: 1.4707
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 1.6276 - val_l

# Inference Model

Training model != Prediction model
we build separate interface models

# Encoder Interface

In [ ]:
encoder_model = Model(encoder_input, encoder_states)

# Decoder Interface

In [ ]:
decoder_state_h = Input(shape = (latent_dim,))
decoder_state_c = Input(shape = (latent_dim,))
decoder_states_inputs = [decoder_state_h, decoder_state_c]

dec_emb2 = Embedding(vocab_size_target, embedding_dim)
dec_emb2 = dec_emb2(decoder_input)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state = decoder_states_inputs
)

decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_input] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)

In [ ]:
reverse_target_index = {i: word for word, i in target_tokenizer.word_index.items()}

def decode_sequence(input_seq):

    states_value = encoder_model.predict(input_seq)

    start_token = target_tokenizer.word_index.get('<start>')

    if start_token is None:
        start_token = target_tokenizer.word_index.get('start')

    end_token = target_tokenizer.word_index.get('<end>')

    if end_token is None:
        end_token = target_tokenizer.word_index.get('end')

    target_seq = np.array([[start_token]])

    stop_condition = False
    decoded_sentence = ""

    while not stop_condition:

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value
        )

        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        sampled_word = reverse_target_index.get(
            sampled_token_index, ''
        )

        if (
            sampled_token_index == end_token or
            len(decoded_sentence.split()) > max_target_len
        ):
            stop_condition = True
        else:
            decoded_sentence += " " + sampled_word

        target_seq = np.array([[sampled_token_index]])

        states_value = [h, c]

    return decoded_sentence


Test

In [ ]:
test_input = "i am happy"
seq = input_tokenizer.texts_to_sequences([test_input])
seq = pad_sequences(seq, maxlen=max_input_len, padding = 'post')

print("Input:", test_input)
print("output:", decode_sequence(seq))

Input: i am happy
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
output:  je suis en train d apprendre
